# Colab runner

Runs the project on a Colab GPU from VS Code.

**Every new session:** run cells 1 and 2 first, in order.

- Code comes from GitHub (always the latest pushed version).
- Big files (`data/`, `outputs/`, `judges/`) live on Google Drive, so they survive when the session ends.

## 1. Connect Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Get the latest code and install it

Paste your GitHub repository address between the quotes on the first line (once).

In [9]:
REPO = "https://github.com/tca-codes/moral-rag-cf.git"   # e.g. "https://github.com/your-username/moral-rag-cf.git"
DRIVE_DIR = "/content/drive/MyDrive/moral-rag-cf"

import os
assert REPO, "Paste your GitHub repository address into REPO on the first line of this cell."
assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not connected. Run cell 1 first."

if not os.path.isdir("/content/moral-rag-cf/.git"):
    !git clone -q {REPO} /content/moral-rag-cf
%cd /content/moral-rag-cf
!git pull -q
!pip install -q -r requirements.txt
!pip install -q -e .

# data, outputs and judges are stored on Drive and linked into the code folder
for name in ["data", "outputs", "judges"]:
    os.makedirs(f"{DRIVE_DIR}/{name}", exist_ok=True)
    if not os.path.exists(name):
        os.symlink(f"{DRIVE_DIR}/{name}", name)

print("\nCode version in use:")
!git log -1 --format="%h  %s  (%cr)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content/moral-rag-cf
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mvrag (pyproject.toml) ... done

Code version in use:
b72bbf2  Save results to Hugging Face instead of Drive  (5 minutes ago)
name, memory.total [MiB]
NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


## 3. Speed test (40 questions, 7B model, uncompressed)

Use this on an L4 or A100. On a T4, remove `--no-4bit` and change 16 to 2.

In [3]:
!python scripts/run_agents.py --split test --n 40 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --out outputs/speedtest.jsonl

40 questions selected from split=test
config.json: 100% 663/663 [00:00<00:00, 3.25MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 23.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 114MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 116MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 159MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 69.0MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.56G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/7.42G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/11.4G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   2% 256M/15.2G [00:04<03:04, 81.2MB/s, 45.0MB/s  ]
Reconstructing (incomplete total...):  15% 2.29G/15.2G [00:11<01:07, 191MB/s,  115MB/s  ]
Reconstructing (incomplete to

In [5]:
%cd /content/moral-rag-cf
!cp --update=none outputs/speedtest.jsonl outputs/agents_test_7B_full.jsonl
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --out outputs/agents_test_7B_full.jsonl
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --questions-per-chunk 16 --plain --out outputs/plain_test_7B_full.jsonl
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_full.jsonl --out outputs/table1_7B_full.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_full.jsonl --out outputs/judge_scores_7B_full.json

/content/moral-rag-cf
3753 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.17it/s]
Done. 0 new questions written to outputs/agents_test_7B_full.jsonl
3753 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.59it/s]
Done. 0 new questions written to outputs/plain_test_7B_full.jsonl
Loading weights: 100% 105/105 [00:00<00:00, 3364.88it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3398.24it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3577.19it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3959.35it/s]
Loading weights: 100% 105/105 [00:00<00:00, 3016.82it/s]
questions: 3753

moral value    plain model  moral agent   change
care               79.53%       87.18%   +7.65%
fairness           85.71%       89.18%   +3.48%
loyalty            88.40%       93.50%   +5.10%
authority          78.72%     

In [7]:
%cd /content/moral-rag-cf
!python scripts/run_aggregator.py --agents outputs/agents_test_7B_full.jsonl --no-4bit --batch-size 16 --limit 200
!python scripts/eval_aggregator.py

/content/moral-rag-cf
200 distinct (question, profile) inputs; 0 rows skipped (no agent answers yet)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 77.15it/s]
  160/200  (77s)
  200/200  (97s)
Done. 200 new outputs written to outputs/aggregator_test_prompt.jsonl

Reference: human-written reply (worker_answer)   rows scored: 214
system           rouge1     rouge2     rougeL  rougeLsum
plain            0.1665     0.0243     0.1319     0.1319
best_agent       0.1482     0.0255     0.1201     0.1201
aggregator       0.1511     0.0263     0.1234     0.1234

Reference: chatbot reply (A)   rows scored: 214
system           rouge1     rouge2     rougeL  rougeLsum
plain            0.2171     0.0551     0.1705     0.1705
best_agent       0.2082     0.0494     0.1594     0.1594
aggregator       0.2112     0.0515     0.1618     0.1618

Saved to outputs/rouge_aggregator_prompt.json


In [8]:
import json
agents = {json.loads(l)["uid"]: json.loads(l) for l in open("outputs/agents_test_7B_full.jsonl")}
for line in open("outputs/aggregator_test_prompt.jsonl").readlines()[:3]:
    r = json.loads(line)
    a = agents[r["uid"]]
    print("QUESTION:", a["question"])
    print("PROFILE (care, fairness, loyalty, authority, sanctity):", r["profile"])
    for value, answer in a["answers"].items():
        print(f"  [{value}] {answer}")
    print("MODERATOR:", r["output"])
    print("=" * 80)

QUESTION: Parents of multiple children: which one do you love more and why?
PROFILE (care, fairness, loyalty, authority, sanctity): [1, 1, 1, 0, 0]
  [care] I don't love one child more than another; each one is special and important in their own way. My job as a parent is to ensure all of them feel loved and protected.
  [fairness] I believe in treating all my children equally and loving them the same, so I don't think it's fair to compare or favor one over the others.
  [loyalty] I love each of my kids equally, but if I had to choose a reason, it's because they all bring something unique to our family, and standing by them all is what matters most.
  [authority] As much as it might seem like I'm playing favorites, I don't love any of my children more than the others. Each one is unique and special in their own way, and they all receive equal love and care according to their needs and the roles they play within our family.
  [sanctity] I don't love any more than the others; each child 

In [9]:
%cd /content/moral-rag-cf
!python scripts/train_judges.py --max-train 1000000 --max-dev 100000 --judges-dir judges/full
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_full.jsonl --judges-dir judges/full --out outputs/table1_7B_fulljudges.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_full.jsonl --judges-dir judges/full --out outputs/judge_scores_7B_fulljudges.json

/content/moral-rag-cf
SocialChem: 355922 rows from data/socialchem/social-chem-101/social-chem-101.v1.0.tsv
  split counts: {'train': 233501, 'dev': 29234, 'test': 29239, 'none': 3913, 'test-extra': 20050, 'dev-extra': 19985, 'analysis': 20000}
[care] train=81117 dev=11092  share labeled good=0.65
config.json: 100% 480/480 [00:00<00:00, 1.94MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 111kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 4.69MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 43.4MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 2.74MB/s]

model.safetensors: downloading bytes:  73% 242M/331M [00:03<00:00, 159MB/s, 20.7MB/s  ]  
model.safetensors: downloading bytes:  94% 311M/331M [00:04<00:00, 58.8MB/s, 24.2MB/s  ]
model.safetensors: downloading bytes: 100% 314M/314M [00:05<00:00, 59.6MB/s, 24.6MB/s  ] ]
model.safetensors: reconstructing file: 100% 331M/331M [00:05<00:00, 62.8MB/s, 28.8MB/s  ]
Loading weights: 100% 101/101 [00:00<00:00, 5442.74it/s]
[transformer

In [10]:
import json
m = json.load(open("judges/full/metrics.json"))
print(f"{'judge':<10}{'accuracy':>10}{'macro_f1':>10}{'majority':>10}{'n_train':>9}{'n_dev':>8}")
for v, r in m.items():
    print(f"{v:<10}{r['accuracy']:>10.3f}{r['macro_f1']:>10.3f}{r['majority_baseline']:>10.3f}{r['n_train']:>9}{r['n']:>8}")
t = json.load(open("outputs/table1_7B_fulljudges.json"))
print(f"\nTable 1 with full judges ({t['n']} questions)")
for v, r in t["rows"].items():
    print(f"{v:<10} plain {r['plain']:.2%}   agent {r['agent']:.2%}   change {r['agent']-r['plain']:+.2%}")
s = json.load(open("outputs/judge_scores_7B_fulljudges.json"))
print(f"\nown judge mean {s['own_mean']:.3f}   other judges mean {s['others_mean']:.3f}")

judge       accuracy  macro_f1  majority  n_train   n_dev
care           0.915     0.904     0.655    81117   11092
fairness       0.874     0.873     0.555    32059    4316
loyalty        0.890     0.881     0.647    34254    4555
authority      0.862     0.857     0.624    19963    2643
sanctity       0.852     0.852     0.516    13196    1743

Table 1 with full judges (3753 questions)
care       plain 92.64%   agent 97.20%   change +4.56%
fairness   plain 82.06%   agent 87.01%   change +4.95%
loyalty    plain 83.34%   agent 89.59%   change +6.25%
authority  plain 78.51%   agent 87.17%   change +8.67%
sanctity   plain 81.50%   agent 88.79%   change +7.29%

own judge mean 0.900   other judges mean 0.889


In [11]:
%cd /content/moral-rag-cf
!python scripts/run_aggregator.py --agents outputs/agents_test_7B_full.jsonl --no-4bit --batch-size 16
!python scripts/eval_aggregator.py

/content/moral-rag-cf
9144 distinct (question, profile) inputs; 0 rows skipped (no agent answers yet)
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 77.68it/s] 
  160/8944  (73s)
  320/8944  (147s)
  480/8944  (221s)
  640/8944  (296s)
  800/8944  (370s)
  960/8944  (443s)
  1120/8944  (516s)
  1280/8944  (592s)
  1440/8944  (669s)
  1600/8944  (744s)
  1760/8944  (821s)
  1920/8944  (900s)
  2080/8944  (977s)
  2240/8944  (1049s)
  2400/8944  (1125s)
  2560/8944  (1200s)
  2720/8944  (1271s)
  2880/8944  (1342s)
  3040/8944  (1418s)
  3200/8944  (1494s)
  3360/8944  (1574s)
  3520/8944  (1651s)
  3680/8944  (1729s)
  3840/8944  (1802s)
  4000/8944  (1878s)
  4160/8944  (1953s)
  4320/8944  (2029s)
  4480/8944  (2105s)
  4640/8944  (2182s)
  4800/8944  (2257s)
  4960/8944  (2333s)
  5120/8944  (2407s)
  5280/8944  (2484s)
  5440/8944  (2562s)
  5600/8944  (2638s)
  5760/8944  (2714s)
  5920/8944  (2789s)
  6080/8944  (2863s)

In [ ]:
%cd /content/moral-rag-cf
!python scripts/sample_candidates.py --n 32 --samples 8 --no-4bit --out-dir outputs/reward_trial
!python scripts/train_agents.py --no-4bit --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial
!python scripts/run_agents.py --split test --n 16 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir outputs/adapters/agents_trial --questions-per-chunk 16 --out outputs/agents_trial_rl.jsonl

In [3]:
import os
for path in ["judges/full/care", "outputs/agents_test_7B_full.jsonl", "outputs/plain_test_7B_full.jsonl"]:
    print("OK     " if os.path.exists(f"/content/moral-rag-cf/{path}") else "MISSING", path)

OK      judges/full/care
OK      outputs/agents_test_7B_full.jsonl
OK      outputs/plain_test_7B_full.jsonl


In [6]:
%cd /content/moral-rag-cf
!cat outputs/reward_trial/summary.json
!python scripts/train_agents.py --no-4bit --grad-checkpointing --values care --max-steps 5 --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial > outputs/train_trial.log 2>&1; tail -n 25 outputs/train_trial.log

/content/moral-rag-cf
{
  "care": {
    "n": 32,
    "mean_candidate_reward": 0.9523533783794846,
    "mean_best_reward": 0.9927396159619093
  },
  "fairness": {
    "n": 32,
    "mean_candidate_reward": 0.861100161797367,
    "mean_best_reward": 0.9622810631990433
  },
  "loyalty": {
    "n": 32,
    "mean_candidate_reward": 0.8677697498642374,
    "mean_best_reward": 0.9751603547483683
  },
  "authority": {
    "n": 32,
    "mean_candidate_reward": 0.8315233875327976,
    "mean_best_reward": 0.9655160121619701
  },
  "sanctity": {
    "n": 32,
    "mean_candidate_reward": 0.8337508497643284,
    "mean_best_reward": 0.945839636027813
  }
}usage: train_agents.py [-h] [--reward-dir REWARD_DIR] [--out-dir OUT_DIR]
                       [--values {care,fairness,loyalty,authority,sanctity} [{care,fairness,loyalty,authority,sanctity} ...]]
                       [--model MODEL] [--epochs EPOCHS]
                       [--batch-size BATCH_SIZE] [--grad-accum GRAD_ACCUM]
                    

In [8]:
!pip uninstall -q -y torchao
%cd /content/moral-rag-cf
!git log -1 --format="Code version: %h %s"
!python scripts/train_agents.py --no-4bit --values care --max-steps 5 --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial > outputs/train_trial.log 2>&1; tail -n 25 outputs/train_trial.log

/content/moral-rag-cf
Code version: bfe9078 Add memory-saving option for advisor training
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2/modeling_qwen2.py", line 380, in forward
    hidden_states = decoder_layer(
        hidden_states,
    ...<5 lines>...
        **kwargs,
    )
  File "/usr/local/lib/python3.13/dist-packages/transformers/modeling_layers.py", line 111, in __call__
    return super().__call__(*args, **kwargs)
           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1783, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1794, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/qwen2/modeling_qwen2.py", line 288, in forward
    hidden_states = self.mlp(hidden_states)

In [11]:
%cd /content/moral-rag-cf
!git pull
!git log -1 --format="Code version: %h %s"
!grep -c "grad-checkpointing" scripts/train_agents.py
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_agents.py --no-4bit --grad-checkpointing --batch-size 4 --grad-accum 4 --values care --max-steps 5 --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial > outputs/train_trial.log 2>&1; tail -n 25 outputs/train_trial.log

/content/moral-rag-cf
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 9 (delta 6), reused 9 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 2.58 KiB | 881.00 KiB/s, done.
From https://github.com/tca-codes/moral-rag-cf
   bfe9078..6264379  main       -> origin/main
Updating bfe9078..6264379
Fast-forward
 notebooks/colab_runner.ipynb | 282 +++++++++++++++++--------------------------
 scripts/train_agents.py      |   5 +-
 src/mvrag/agg_train.py       |  13 +-
 3 files changed, 127 insertions(+), 173 deletions(-)
Code version: 6264379 Add memory-saving option (files)
1
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

[care] training on 32 winning answers
Loading weights: 100%|██████████| 339/339 [00:03<00:00, 88.39it/s] 
32 examples, 4 steps in total; running steps 0 to 4
  step 4/4  loss 0.6649  3.03s/step  about 0.0h left for the full run  peak GPU 16.0GB
[

In [12]:
%cd /content/moral-rag-cf
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_agents.py --no-4bit --grad-checkpointing --batch-size 4 --grad-accum 4 --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial > outputs/train_trial.log 2>&1; tail -n 6 outputs/train_trial.log
!python scripts/run_agents.py --split test --n 16 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir outputs/adapters/agents_trial --questions-per-chunk 16 --out outputs/agents_trial_rl.jsonl

import json
r = json.loads(open("outputs/agents_trial_rl.jsonl").readline())
print("\nQUESTION:", r["question"])
for value, answer in r["answers"].items():
    print(f"[{value}] {answer}")

/content/moral-rag-cf

[sanctity] training on 32 winning answers
Loading weights: 100%|██████████| 339/339 [00:03<00:00, 87.64it/s] 
32 examples, 4 steps in total; running steps 0 to 4
  step 4/4  loss 0.7590  3.06s/step  about 0.0h left for the full run  peak GPU 16.1GB
[sanctity] saved to outputs/adapters/agents_trial/sanctity: step 4/4
16 questions selected from split=test
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 76.79it/s]
  16/16 new questions  (28s)
Done. 16 new questions written to outputs/agents_trial_rl.jsonl

QUESTION: Parents of multiple children: which one do you love more and why?
[care] I believe it's important to love each child equally and focus on their unique needs and well-being rather than comparing them.
[fairness] I believe in treating all my children equally and loving them the same, so I don't think it's fair to compare or favor one over the others.
[loyalty] I love each of my kids equally, but 

In [4]:
!pip uninstall -q -y torchao
%cd /content/moral-rag-cf
!python scripts/sample_candidates.py --n 64 --samples 8 --no-4bit --batch-size 32 --values care --out-dir outputs/reward_trial_a100
!python scripts/train_agents.py --no-4bit --batch-size 8 --grad-accum 2 --values care --max-steps 3 --reward-dir outputs/reward_trial --out-dir outputs/adapters/agents_trial_a100 > outputs/train_trial_a100.log 2>&1; tail -n 4 outputs/train_trial_a100.log

/content/moral-rag-cf
64 questions from split=train, 8 candidates each, values=['care']
config.json: 100% 663/663 [00:00<00:00, 3.41MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 25.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 121MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 114MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 162MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 90.1MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/7.81G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   3% 399M/15.2G [00:02<01:01, 242MB/s, 62.1MB/s  ] 
Reconstructing (incomplete total...):   3% 512M/15.2G [00:02<00:45, 3

In [5]:
%cd /content/moral-rag-cf
import json
!python scripts/train_judges.py --base-model roberta-large --max-train 1000000 --max-dev 100000 --judges-dir judges/large
acc = lambda path: sum(v["accuracy"] for v in json.load(open(path)).values()) / 5
JUDGES = "judges/large" if acc("judges/large/metrics.json") > acc("judges/full/metrics.json") + 0.005 else "judges/full"
print("Using judges:", JUDGES, "| large:", round(acc("judges/large/metrics.json"), 4), "| current:", round(acc("judges/full/metrics.json"), 4))

/content/moral-rag-cf
SocialChem: 355922 rows from data/socialchem/social-chem-101/social-chem-101.v1.0.tsv
  split counts: {'train': 233501, 'dev': 29234, 'test': 29239, 'none': 3913, 'test-extra': 20050, 'dev-extra': 19985, 'analysis': 20000}
[care] train=81117 dev=11092  share labeled good=0.65
config.json: 100% 482/482 [00:00<00:00, 2.08MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 123kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 9.66MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 1.86MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 1.64MB/s]

model.safetensors: downloading bytes:  13% 188M/1.42G [00:01<00:06, 197MB/s, 12.9MB/s  ]  
model.safetensors: reconstructing file:   9% 134M/1.42G [00:01<00:16, 77.6MB/s]
model.safetensors: downloading bytes:  25% 355M/1.42G [00:01<00:02, 360MB/s, 26.0MB/s  ] ]
model.safetensors: reconstructing file:  28% 402M/1.42G [00:02<00:03, 263MB/s, 25.4MB/s  ]
model.safetensors: downloading bytes:  45% 636M/1.42G [00:02<00:01, 502MB/s, 

In [6]:
%cd /content/moral-rag-cf
!git pull -q
!git log -1 --format="Code version: %h %s"
import json
full = json.load(open("judges/full/metrics.json")); large = json.load(open("judges/large/metrics.json"))
print(f"\n{'judge':<10}{'current':>9}{'large':>9}{'majority':>10}")
for v in full:
    print(f"{v:<10}{full[v]['accuracy']:>9.3f}{large[v]['accuracy']:>9.3f}{full[v]['majority_baseline']:>10.3f}")

bad = [v for v in full if large[v]["accuracy"] < full[v]["accuracy"]]
print("\nRetraining large judges for:", bad)
if bad:
    !python scripts/train_judges.py --base-model roberta-large --max-train 1000000 --max-dev 100000 --judges-dir judges/large --force --lr 1e-5 --warmup 0.06 --values {" ".join(bad)}
!python scripts/pick_judges.py --candidates judges/full judges/large --out judges/best

/content/moral-rag-cf
Code version: b2df88f Add judge warmup and per-value judge picking

judge       current    large  majority
care          0.915    0.941     0.655
fairness      0.874    0.555     0.555
loyalty       0.890    0.911     0.647
authority     0.862    0.877     0.624
sanctity      0.852    0.888     0.516

Retraining large judges for: ['fairness']
SocialChem: 355922 rows from data/socialchem/social-chem-101/social-chem-101.v1.0.tsv
  split counts: {'train': 233501, 'dev': 29234, 'test': 29239, 'none': 3913, 'test-extra': 20050, 'dev-extra': 19985, 'analysis': 20000}
[fairness] train=32059 dev=4316  share labeled good=0.57
Loading weights: 100% 389/389 [00:00<00:00, 5683.58it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
l

In [7]:
!pip uninstall -q -y torchao
%cd /content/moral-rag-cf
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
J = "judges/best"

# 1. Eight candidate answers per question, scored by the judges: 4,000 questions, then extended to 8,000
!python scripts/sample_candidates.py --n 4000 --samples 8 --no-4bit --batch-size 32 --judges-dir {J}
!python scripts/sample_candidates.py --n 8000 --samples 8 --no-4bit --batch-size 32 --judges-dir {J}

# 2. Train the five advisors on their best-scoring answers
!python scripts/train_agents.py --no-4bit --batch-size 8 --grad-accum 2

# 3. Trained advisors answer the whole test set
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir outputs/adapters/agents --questions-per-chunk 64 --out outputs/agents_test_7B_rl.jsonl

# 4. Table 1 for the instructed and the trained advisors, all scored by the new judges
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_full.jsonl --judges-dir {J} --out outputs/table1_prompted_best.json
!python scripts/table1.py --plain outputs/plain_test_7B_full.jsonl --agents outputs/agents_test_7B_rl.jsonl --judges-dir {J} --out outputs/table1_rl_best.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_full.jsonl --judges-dir {J} --out outputs/judge_scores_prompted_best.json
!python scripts/score_agents.py --agents outputs/agents_test_7B_rl.jsonl --judges-dir {J} --out outputs/judge_scores_rl_best.json

/content/moral-rag-cf
4000 questions from split=train, 8 candidates each, values=['care', 'fairness', 'loyalty', 'authority', 'sanctity']
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 339/339 [00:04<00:00, 73.70it/s]
Loading weights: 100% 393/393 [00:00<00:00, 3235.71it/s]
  [care] 320/4000 new questions  (107s)
  [care] 640/4000 new questions  (201s)
  [care] 960/4000 new questions  (293s)
  [care] 1280/4000 new questions  (388s)
  [care] 1600/4000 new questions  (553s)
  [care] 1920/4000 new questions  (654s)
  [care] 2240/4000 new questions  (749s)
  [care] 2560/4000 new questions  (843s)
  [care] 2880/4000 new questions  (946s)
  [care] 3200/4000 new questions  (1046s)
  [care] 3520/4000 new questions  (1139s)
  [care] 3840/4000 new questions  (1234s)
  [care] 4000/4000 new questions  (1282s)
Loading weights: 100% 393/393 [00:13<00:00, 28.73it/s]  
  [fairness] 320/4000 new questions  (123s)
  [fairness] 640/4000 new questions  (217s)
  [fai

In [8]:
%cd /content/moral-rag-cf
import os
V = ["care", "fairness", "loyalty", "authority", "sanctity"]
lines = lambda p: sum(1 for _ in open(p)) if os.path.exists(p) else "missing"
print("Candidate files (questions saved):")
for v in V:
    print(f"  {v:<10}", lines(f"outputs/reward/{v}.jsonl"))
print("Trained-advisor test answers:", lines("outputs/agents_test_7B_rl.jsonl"))
print("\nJudge model files present:")
for d in ["judges/full", "judges/large", "judges/best"]:
    ok = [v for v in V if os.path.exists(f"{d}/{v}/model.safetensors")]
    print(f"  {d:<13} {len(ok)}/5")
!df -h /content/drive | tail -1
!du -sh /content/drive/MyDrive/moral-rag-cf/judges/* /content/drive/MyDrive/moral-rag-cf/outputs /content/drive/MyDrive/moral-rag-cf/data 2>/dev/null

/content/moral-rag-cf
Candidate files (questions saved):
  care       4000
  fairness   4000
  loyalty    4000
  authority  4000
  sanctity   4000
Trained-advisor test answers: 3753

Judge model files present:
  judges/full   5/5
  judges/large  4/5
  judges/best   0/5
drive           236G   48G  188G  21% /content/drive
317M	/content/drive/MyDrive/moral-rag-cf/judges/authority
18M	/content/drive/MyDrive/moral-rag-cf/judges/best
317M	/content/drive/MyDrive/moral-rag-cf/judges/care
317M	/content/drive/MyDrive/moral-rag-cf/judges/fairness
1.6G	/content/drive/MyDrive/moral-rag-cf/judges/full
5.4G	/content/drive/MyDrive/moral-rag-cf/judges/large
317M	/content/drive/MyDrive/moral-rag-cf/judges/loyalty
1.5K	/content/drive/MyDrive/moral-rag-cf/judges/metrics.json
317M	/content/drive/MyDrive/moral-rag-cf/judges/sanctity
488M	/content/drive/MyDrive/moral-rag-cf/outputs
276M	/content/drive/MyDrive/moral-rag-cf/data


In [9]:
%cd /content/moral-rag-cf
import os, glob
V = ["care", "fairness", "loyalty", "authority", "sanctity"]
lines = lambda p: sum(1 for _ in open(p)) if os.path.exists(p) else "MISSING"
print("Which large judge is missing:", [v for v in V if not os.path.exists(f"judges/large/{v}/model.safetensors")])
print("Large judge folder contents (first value):", sorted(os.listdir(f"judges/large/{V[0]}")))
print("\nTrained advisors (adapter weights present):")
for v in V:
    print(f"  {v:<10}", os.path.exists(f"outputs/adapters/agents/{v}/adapter_model.safetensors"))
print("\nResult files (expected count):")
for f, n in [("agents_test_7B_full.jsonl", 3753), ("plain_test_7B_full.jsonl", 3753), ("agents_test_7B_rl.jsonl", 3753),
             ("aggregator_test_prompt.jsonl", 9144), ("baseline_llama2_7b.jsonl", 7796),
             ("baseline_llama2_13b.jsonl", 7796), ("baseline_llama31_8b.jsonl", 7796)]:
    print(f"  {f:<32} {lines('outputs/' + f)}  (expected {n})")

/content/moral-rag-cf
Which large judge is missing: ['sanctity']
Large judge folder contents (first value): ['config.json', 'metrics.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']

Trained advisors (adapter weights present):
  care       True
  fairness   True
  loyalty    True
  authority  True
  sanctity   True

Result files (expected count):
  agents_test_7B_full.jsonl        3753  (expected 3753)
  plain_test_7B_full.jsonl         3753  (expected 3753)
  agents_test_7B_rl.jsonl          3753  (expected 3753)
  aggregator_test_prompt.jsonl     9144  (expected 9144)
  baseline_llama2_7b.jsonl         7796  (expected 7796)
  baseline_llama2_13b.jsonl        7796  (expected 7796)
  baseline_llama31_8b.jsonl        7796  (expected 7796)


In [ ]:
!pip uninstall -q -y torchao
%cd /content/moral-rag-cf
!git pull -q
!git log -1 --format="Code version: %h %s"
!pip install -q -e .
import os, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
W, D = "/content/work", "/content/drive/MyDrive/moral-rag-cf"
J = f"{W}/judges/large"

def save(rel):
    """Copy a result from the local disk to Drive, then check the Drive copy matches."""
    parent = os.path.dirname(f"{D}/{rel}")
    os.makedirs(parent, exist_ok=True)
    subprocess.run(["rsync", "-a", f"{W}/{rel}", parent + "/"], check=True)
    same = subprocess.run(["diff", "-rq", f"{W}/{rel}", f"{D}/{rel}"], capture_output=True).returncode == 0
    print(("SAVED    " if same else "PROBLEM  ") + rel)

# 0. Local working copies of what's already saved on Drive (lets a rerun resume)
!mkdir -p {W}/outputs {J}
!rsync -a {D}/judges/large/ {J}/
!rsync -a {D}/outputs/reward {D}/outputs/adapters {W}/outputs/
!cp -n {D}/outputs/agents_test_7B_rl*.jsonl {W}/outputs/ 2>/dev/null; true

# 1. Retrain the lost sanctity judge (same settings as the original), save it
if not os.path.exists(f"{J}/sanctity/model.safetensors"):
    !python scripts/train_judges.py --base-model roberta-large --max-train 1000000 --max-dev 100000 --judges-dir {J} --values sanctity --force
save("judges/large/sanctity")
!rm -rf {D}/judges/best

# 2. Candidates for questions 4,001 to 8,000 (the lost batch)
!python scripts/sample_candidates.py --n 8000 --samples 8 --no-4bit --batch-size 32 --judges-dir {J} --out-dir {W}/outputs/reward
save("outputs/reward")

# 3. Advisors trained on 8,000 questions
!python scripts/train_agents.py --no-4bit --batch-size 8 --grad-accum 2 --reward-dir {W}/outputs/reward --out-dir {W}/outputs/adapters/agents_8k
save("outputs/adapters/agents_8k")

# 4. They answer the test set
!python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir {W}/outputs/adapters/agents_8k --questions-per-chunk 64 --out {W}/outputs/agents_test_7B_rl8k.jsonl
save("outputs/agents_test_7B_rl8k.jsonl")

# 5. Table 1 with the same judges: instructed, trained on 4,000, trained on 8,000
for name, agents in [("prompted", f"{D}/outputs/agents_test_7B_full.jsonl"),
                     ("rl4k", f"{D}/outputs/agents_test_7B_rl.jsonl"),
                     ("rl8k", f"{W}/outputs/agents_test_7B_rl8k.jsonl")]:
    print(f"\n######## {name} ########")
    !python scripts/table1.py --plain {D}/outputs/plain_test_7B_full.jsonl --agents {agents} --judges-dir {J} --out {W}/outputs/table1_{name}.json
    !python scripts/score_agents.py --agents {agents} --judges-dir {J} --out {W}/outputs/judge_scores_{name}.json
    save(f"outputs/table1_{name}.json"); save(f"outputs/judge_scores_{name}.json")

# 6. Overnight: candidates for all 28,704 training questions, saved after each block
for n in [12000, 16000, 20000, 24000, 100000]:
    !python scripts/sample_candidates.py --n {n} --samples 8 --no-4bit --batch-size 32 --judges-dir {J} --out-dir {W}/outputs/reward
    save("outputs/reward")

# 7. Advisors trained on all questions, plus 2,000 and 16,000 for the size comparison
for tag, limit in [("all", ""), ("2k", "--limit-questions 2000"), ("16k", "--limit-questions 16000")]:
    !python scripts/train_agents.py --no-4bit --batch-size 8 --grad-accum 2 {limit} --reward-dir {W}/outputs/reward --out-dir {W}/outputs/adapters/agents_{tag}
    save(f"outputs/adapters/agents_{tag}")
    !python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir {W}/outputs/adapters/agents_{tag} --questions-per-chunk 64 --out {W}/outputs/agents_test_7B_rl{tag}.jsonl
    save(f"outputs/agents_test_7B_rl{tag}.jsonl")
    print(f"\n######## rl{tag} ########")
    !python scripts/table1.py --plain {D}/outputs/plain_test_7B_full.jsonl --agents {W}/outputs/agents_test_7B_rl{tag}.jsonl --judges-dir {J} --out {W}/outputs/table1_rl{tag}.json
    !python scripts/score_agents.py --agents {W}/outputs/agents_test_7B_rl{tag}.jsonl --judges-dir {J} --out {W}/outputs/judge_scores_rl{tag}.json
    save(f"outputs/table1_rl{tag}.json"); save(f"outputs/judge_scores_rl{tag}.json")

from google.colab import drive
drive.flush_and_unmount()
print("ALL DONE")

/content/moral-rag-cf
Code version: 167972f Add RAG and counterfactual extension
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for mvrag (pyproject.toml) ... done
SocialChem: 355922 rows from data/socialchem/social-chem-101/social-chem-101.v1.0.tsv
  split counts: {'train': 233501, 'dev': 29234, 'test': 29239, 'none': 3913, 'test-extra': 20050, 'dev-extra': 19985, 'analysis': 20000}
[sanctity] train=13196 dev=1743  share labeled good=0.51
Loading weights: 100% 389/389 [00:00<00:00, 6616.99it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias     

In [ ]:
import getpass, os, subprocess
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face token (with Write access): ").strip()
!pip uninstall -q -y torchao
%cd /content/moral-rag-cf
!git log -1 --format="Code version: %h %s"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
W, D = "/content/work", "/content/drive/MyDrive/moral-rag-cf"
J, JF = f"{W}/judges/large", f"{D}/judges/full"

def hub(action, root, *paths):
    r = subprocess.run(["python", "scripts/hub.py", action, "--root", root, *paths])
    if r.returncode != 0:
        raise RuntimeError("Hub step failed; stopping so nothing is lost.")

# 1. Copy everything that survived on Drive to the Hub
hub("push", D, "judges/full", "judges/large/care", "judges/large/fairness", "judges/large/loyalty", "judges/large/authority",
    "outputs/adapters/agents", "outputs/adapters/agents_8k", "outputs/reward",
    "outputs/agents_test_7B_full.jsonl", "outputs/plain_test_7B_full.jsonl",
    "outputs/agents_test_7B_rl.jsonl", "outputs/agents_test_7B_rl8k.jsonl", "outputs/aggregator_test_prompt.jsonl",
    "outputs/baseline_llama2_7b.jsonl", "outputs/baseline_llama2_13b.jsonl", "outputs/baseline_llama31_8b.jsonl")

# 2. Local working copies; retrain the lost sanctity judge and save it
!mkdir -p {W}/outputs {J}
!rsync -a {D}/judges/large/ {J}/
!rsync -a {D}/outputs/reward {W}/outputs/
if not os.path.exists(f"{J}/sanctity/model.safetensors"):
    !python scripts/train_judges.py --base-model roberta-large --max-train 1000000 --max-dev 100000 --judges-dir {J} --values sanctity --force
hub("push", W, "judges/large/sanctity")

# 3. Candidates for the remaining questions (16,001 to 28,704), saved after each block
for n in [20000, 24000, 100000]:
    !python scripts/sample_candidates.py --n {n} --samples 8 --no-4bit --batch-size 64 --judges-dir {J} --out-dir {W}/outputs/reward
    hub("push", W, "outputs/reward")

# 4. Advisors trained on all questions, 2,000 and 16,000; their test answers
for tag, limit in [("all", ""), ("2k", "--limit-questions 2000"), ("16k", "--limit-questions 16000")]:
    !python scripts/train_agents.py --no-4bit --batch-size 8 --grad-accum 2 {limit} --reward-dir {W}/outputs/reward --out-dir {W}/outputs/adapters/agents_{tag}
    hub("push", W, f"outputs/adapters/agents_{tag}")
    !python scripts/run_agents.py --split test --n 4000 --max-new-tokens 60 --model Qwen/Qwen2.5-7B-Instruct --no-4bit --adapters-dir {W}/outputs/adapters/agents_{tag} --questions-per-chunk 128 --out {W}/outputs/agents_test_7B_rl{tag}.jsonl
    hub("push", W, f"outputs/agents_test_7B_rl{tag}.jsonl")

# 5. Every Table 1 with the same final judges, plus the independent DistilRoBERTa check
versions = [("prompted", f"{D}/outputs/agents_test_7B_full.jsonl"), ("rl2k", f"{W}/outputs/agents_test_7B_rl2k.jsonl"),
            ("rl4k", f"{D}/outputs/agents_test_7B_rl.jsonl"), ("rl8k", f"{D}/outputs/agents_test_7B_rl8k.jsonl"),
            ("rl16k", f"{W}/outputs/agents_test_7B_rl16k.jsonl"), ("rlall", f"{W}/outputs/agents_test_7B_rlall.jsonl")]
for tag, f in versions:
    print(f"\n######## {tag} ########")
    !python scripts/table1.py --plain {D}/outputs/plain_test_7B_full.jsonl --agents {f} --judges-dir {J} --out {W}/outputs/table1_{tag}.json
    !python scripts/score_agents.py --agents {f} --judges-dir {J} --out {W}/outputs/judge_scores_{tag}.json
    print(f"######## {tag}, independent judges ########")
    !python scripts/table1.py --plain {D}/outputs/plain_test_7B_full.jsonl --agents {f} --judges-dir {JF} --out {W}/outputs/table1_{tag}_distil.json
    hub("push", W, f"outputs/table1_{tag}.json", f"outputs/judge_scores_{tag}.json", f"outputs/table1_{tag}_distil.json")

print("\nEverything stored on the Hub:")
hub("list", W)
print("ALL DONE")

/content/moral-rag-cf
Code version: b72bbf2 Save results to Hugging Face instead of Drive
SocialChem: 355922 rows from data/socialchem/social-chem-101/social-chem-101.v1.0.tsv
  split counts: {'train': 233501, 'dev': 29234, 'test': 29239, 'none': 3913, 'test-extra': 20050, 'dev-extra': 19985, 'analysis': 20000}
[sanctity] train=13196 dev=1743  share labeled good=0.51
config.json: 100% 482/482 [00:00<00:00, 5.00MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 327kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 21.8MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 88.5MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 29.7MB/s]

model.safetensors: downloading bytes:  63% 892M/1.42G [00:02<00:01, 521MB/s, 73.5MB/s  ]  
model.safetensors: downloading bytes:  67% 949M/1.42G [00:03<00:01, 306MB/s, 79.5MB/s  ]
model.safetensors: downloading bytes: 100% 952M/952M [00:03<00:00, 256MB/s, 79.2MB/s  ]   ]
model.safetensors: reconstructing file: 100% 1.42G/1.42G [00:03<00:00, 381MB/s,  129MB/

## When you are done

Disconnect the runtime so it stops using compute units: open the Command Palette and run **Colab: Remove Server**.